# Appendix E.2 – Sonority Notebook

This annotated Jupyter notebook accompanies the analytical discussion of sonority and harmony in Chapter 2 of the thesis. It documents the computational procedures used to extract, classify, and quantify vertical sonorities in a corpus of works from the Eton Choirbook, encoded in machine‑readable format (MusicXML).

The notebook serves two principal purposes. First, it provides methodological transparency by making explicit the analytical steps that underpin the quantitative observations presented in the main text. Secondly, it functions as an exploratory research tool, enabling patterns of sonority usage to be examined across repertory at a scale that would be impractical using manual analysis alone.

Sonorities are extracted using the *CRIM Intervals* toolkit and are initially represented as unordered collections of intervals above the bass. These raw representations are then normalised in order to facilitate stylistic comparison. Octave duplications are removed, compound intervals are reduced to their simple equivalents, and only sonorities containing at least three distinct pitch‑classes above the bass are treated as structurally triadic. Two‑note sonorities (diads) and other reduced textures are retained in the dataset but classified separately.

The resulting sonorities are grouped into broad contrapuntal categories (5/3, 6/3, 6/4, and Other). These categories are not intended to imply functional harmony in a later tonal sense, but rather to provide a historically informed and analytically tractable means of comparing vertical intervallic structures within the Eton repertory.

All code cells are preceded by brief explanatory commentary.


## 1) Imports

Run this cell first. If you’re working on TLJH/JupyterHub where CRIM Intervals is already installed, you should be set.

In [1]:
import os
import glob
import pandas as pd

import crim_intervals
from crim_intervals.main_objs import CorpusBase
import crim_intervals.corpus_tools as corpus_tools

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

print("crim_intervals version:", getattr(crim_intervals, "__version__", "unknown"))


crim_intervals version: 2.0.62


## 2) Point to your corpus folder

By default, this expects your MusicXML files in:

- `Music_Files/Eton_Corpus/*.musicxml`

Adjust `CORPUS_GLOB` if needed.

In [2]:
CORPUS_GLOB = "Music_Files/Eton_Corpus/Reduced_Sections/*.musicxml"

file_list = sorted(glob.glob(CORPUS_GLOB))
print(f"Found {len(file_list)} files.")
file_list[:10]


Found 63 files.


['Music_Files/Eton_Corpus/Reduced_Sections/e10_sturton_gaude_virgo_mater_cristi_reduced1.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e10_sturton_gaude_virgo_mater_cristi_reduced2.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e10_sturton_gaude_virgo_mater_cristi_reduced3.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e10_sturton_gaude_virgo_mater_cristi_reduced4.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced1.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced2.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced3.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced4.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced5.musicxml',
 'Music_Files/Eton_Corpus/Reduced_Sections/e17_horwood_salve_regina_reduced6.musicxml']

## 3) Build a CRIM `CorpusBase`

`CorpusBase` takes a list of file paths. Each file should be parseable by music21 (used under the hood).

In [3]:
if len(file_list) == 0:
    raise FileNotFoundError(
        "No files found. Update CORPUS_GLOB to point at your MusicXML corpus."
    )

corpus = CorpusBase(file_list)
corpus

## 4) Generate sonority n-grams

The Corpus Tools tutorial documents:

`corpus_sonority_ngrams(corpus, ngram_length=4, metadata_choice=True, include_offset=False, include_progress=True, compound=True, sort=False, minimum_beat_strength=0.0)`

Key variables:
- `ngram_length`: how many successive sonorities per n-gram
- `include_progress`: include a 'Progress' column to help track position in a work
- `compound`: keep compound intervals (True) or reduce them (False)
- `sort`: if True, sorts intervals largest→smallest and removes unison (useful for normalising voicings)
- `minimum_beat_strength`: filter out weak metrical positions (0.0 = keep all)

Start with the defaults, then tweak.

In [4]:
import warnings
import contextlib
import io

# Suppress noisy FutureWarnings emitted inside crim_intervals (e.g., pandas chained assignment warnings)
warnings.filterwarnings('ignore', category=FutureWarning, module='crim_intervals')

# --- Parameters you’ll likely want to tweak ---
NGRAM_LENGTH = 1
INCLUDE_OFFSET = False          # True adds raw offsets (can be useful for alignment/debugging)
INCLUDE_PROGRESS = False
COMPOUND = True                 # keep compound intervals
SORT_INTERVALS = False          # set True to normalise sonorities by sorting intervals
MIN_BEAT_STRENGTH = 0.0125         # raise (e.g., 0.25, 0.5) to ignore weak beats

with contextlib.redirect_stderr(io.StringIO()):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        son_ng = corpus_tools.corpus_sonority_ngrams(
            corpus,
            ngram_length=NGRAM_LENGTH,
            metadata_choice=True,
            include_offset=INCLUDE_OFFSET,
            include_progress=INCLUDE_PROGRESS,
            compound=COMPOUND,
            sort=SORT_INTERVALS,
            minimum_beat_strength=MIN_BEAT_STRENGTH,
            )

son_ng.head()

,Measure,Beat,Composer,Title,Date,Sonority,Low Line,Low_Sonority
0,1.0,1.0,Edmund Sturton,Gaude virgo mater Christi,None,"(8/1,)","(M3,)",M3_8/1
1,1.0,3.0,Edmund Sturton,Gaude virgo mater Christi,None,"(5/1,)","(Held,)",Held_5/1
2,2.0,1.0,Edmund Sturton,Gaude virgo mater Christi,None,"(5/1,)","(Held,)",Held_5/1
3,2.0,2.5,Edmund Sturton,Gaude virgo mater Christi,None,"(3/1,)","(m2,)",m2_3/1
4,2.0,3.0,Edmund Sturton,Gaude virgo mater Christi,None,"(3/1,)","(-m3,)",-m3_3/1


## 5) Make a frequency table (counts + %)

`corpus_sonority_ngrams` returns one row per observed n-gram instance. To count “types” (unique patterns), we group by the n-gram column.

**Important:** the n-gram column name can vary slightly depending on your CRIM Intervals version. The helper below finds it robustly.

### 5a) Collapse sonorities to **5/3**, **6/3**, **6/4**, or **Other**

This helper reduces octave duplications (e.g., `8`/`15`) and groups related sonorities so that, for example,
`8/5/1` and `5/3/1` can be counted together as **5/3**.


In [5]:
import re

def _reduce_to_simple_figure_numbers(nums):
    """Reduce compound figures (e.g., 10->3, 12->5, 15->1) into the 1–7 range."""
    reduced = [((n - 1) % 7) + 1 for n in nums if isinstance(n, (int, float))]
    return reduced

def simplify_sonority_value(val):
    """Map a sonority encoding to one of: '5/3', '6/3', '6/4', 'Other'.

    Works with values that look like: (8/5/1,), ('8/5/1',), Fraction-like, or plain strings.
    """
    # Unwrap common container types from corpus_tools
    if isinstance(val, (list, tuple)) and len(val) == 1:
        val = val[0]

    s = str(val)
    nums = [int(x) for x in re.findall(r"\d+", s)]
    if not nums:
        return 'Other'

    reduced = _reduce_to_simple_figure_numbers(nums)

    # Collapse octave duplication already handled by reduction; now work with a set of simple figures
    simple_set = set(reduced)

    # Exclude diads (and singletons): only label 5/3, 6/3, 6/4 if there are at least 3 distinct tones incl. bass (1)
    if len(simple_set) < 3:
        return 'Other'

    # Triadic-ish classification
    if {1, 4, 6}.issubset(simple_set):
        return '6/4'
    if {1, 3, 6}.issubset(simple_set):
        return '6/3'
    if {1, 3, 5}.issubset(simple_set):
        return '5/3'
    return 'Other'

# Add a simplified column you can count instead of raw 'Sonority'
son_ng = son_ng.copy()
son_ng['Sonority_simple'] = son_ng['Sonority'].apply(simplify_sonority_value)
son_ng[['Sonority', 'Sonority_simple']].head(10)


,Sonority,Sonority_simple
0,"(8/1,)",Other
1,"(5/1,)",Other
2,"(5/1,)",Other
3,"(3/1,)",Other
4,"(3/1,)",Other
5,"(6/1,)",Other
6,"(3/1,)",Other
7,"(1,)",Other
8,"(3/1,)",Other
9,"(6/1,)",Other


### 5b) Choose what to count (raw vs simplified)

Set `PAYLOAD_COL` to control what the n-grams are built from:

- `"Sonority_simple"` (recommended): collapsed into **5/3**, **6/3**, **6/4**, **Other**.
- `"Sonority"`: the raw CRIM sonority figure string.
- `"Low_Sonority"`: the *lowest* vertical interval in each sonority (useful for some reductionist views).

`NGRAM_LENGTH` controls how many consecutive sonorities form each pattern (1 = single sonorities; 2 = bigrams; etc.).


In [6]:
# Choose your payload column:
PAYLOAD_COL = "Sonority_simple"  # "Sonority" for raw figures, or "Low_Sonority"
NGRAM_LENGTH = 1               # set to whatever you want

# Sort events into musical order
son_sorted = son_ng.sort_values(["Title", "Measure", "Beat"]).copy()


### 5c) Build n-grams from a single column

This helper turns a column of sonorities into a list of n-gram tuples. It drops missing values so gaps don’t create artificial patterns.


In [7]:
def make_ngrams(series, n):
    """Return list of n-gram tuples from a pandas Series (dropping NaNs)."""
    vals = [v for v in series.tolist() if pd.notna(v)]
    return [tuple(vals[i:i+n]) for i in range(len(vals) - n + 1)]

# Build a long table of n-gram instances
rows = []
for (composer, title), df_piece in son_sorted.groupby(["Composer", "Title"], dropna=False):
    grams = make_ngrams(df_piece[PAYLOAD_COL], NGRAM_LENGTH)
    rows.extend([{"Composer": composer, "Title": title, "Ngram": g} for g in grams])

ngram_instances = pd.DataFrame(rows)
print("N-gram instances:", len(ngram_instances))
ngram_instances.head()

N-gram instances: 10061


,Composer,Title,Ngram
0,Edmund Sturton,Gaude virgo mater Christi,"(Other,)"
1,Edmund Sturton,Gaude virgo mater Christi,"(5/3,)"
2,Edmund Sturton,Gaude virgo mater Christi,"(Other,)"
3,Edmund Sturton,Gaude virgo mater Christi,"(Other,)"
4,Edmund Sturton,Gaude virgo mater Christi,"(Other,)"


### 5d) Frequency table

Here we count how often each **unique n-gram type** occurs in the corpus, then compute percentages and show the top results.


In [8]:
TOP_N = 30

freq = (
    ngram_instances.groupby("Ngram")
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

total = int(freq["Count"].sum())
freq["Percent"] = (freq["Count"] / total * 100).round(2)

freq.head(TOP_N)

,Ngram,Count,Percent
3,"(Other,)",7651,76.05
0,"(5/3,)",1520,15.11
1,"(6/3,)",710,7.06
2,"(6/4,)",180,1.79


## 6) (Optional) Export results

Write the instance-level table and the frequency table to CSV for use elsewhere.

In [ ]:
OUT_DIR = "Outputs"
os.makedirs(OUT_DIR, exist_ok=True)

son_ng_path = os.path.join(OUT_DIR, f"sonority_ngrams_instances_n{NGRAM_LENGTH}.csv")
freq_path = os.path.join(OUT_DIR, f"sonority_ngrams_frequency_n{NGRAM_LENGTH}.csv")

son_ng.to_csv(son_ng_path, index=False)
freq.to_csv(freq_path, index=False)

son_ng_path, freq_path

In [ ]:
!jupyter nbconvert --to html "Eton Recon Aid for Appendix.ipynb"